<style>
.jp-RenderedHTMLCommon { color: #EAEAEA; }
.manus-title {
  background: linear-gradient(135deg, #111827, #1F2937);
  border: 1px solid #374151;
  border-radius: 18px;
  padding: 22px 26px;
  margin: 10px 0 18px 0;
  box-shadow: 0 8px 28px rgba(0,0,0,.28);
}
.manus-title h1 { color: #F9FAFB; margin: 0; font-size: 30px; }
.manus-title p { color: #D1D5DB; margin: 8px 0 0 0; font-size: 15px; }
.callout {
  border-radius: 14px;
  padding: 14px 16px;
  margin: 14px 0;
  border-left: 5px solid;
  background: #111827;
}
.callout strong { color: #F9FAFB; }
.info { border-color: #60A5FA; background: #0F172A; }
.good { border-color: #34D399; background: #052E2B; }
.bad { border-color: #F87171; background: #2A1010; }
.warn { border-color: #FBBF24; background: #2A2108; }
.key { border-color: #A78BFA; background: #1E1633; }
table { width: 100%; border-collapse: collapse; font-size: 14px; }
th { background: #1F2937; color: #F9FAFB; padding: 10px; border: 1px solid #374151; }
td { padding: 10px; border: 1px solid #374151; vertical-align: top; }
tr:nth-child(even) td { background: #111827; }
tr:nth-child(odd) td { background: #0B1120; }
code { color: #FDE68A; background: #1F2937; padding: 2px 5px; border-radius: 6px; }
pre { background: #0B1020 !important; border: 1px solid #374151; border-radius: 12px; padding: 12px; }
</style>

<div class="manus-title">
  <h1>🧠 Manus — Context Engineering for AI Agents</h1>
  <p>Dark-theme engineer notes for LangGraph, Playwright bug-fix agents, Career Radar, MCP, and production agent systems.</p>
</div>

## 🎯 Core Thesis

<div class="callout key">
<strong>Production Agent ≠ LLM + Tools</strong><br>
Production Agent = <strong>Model + Context Architecture + Memory + Error Feedback + Attention Control</strong>
</div>

A strong model helps, but production-grade agents mainly depend on how you design context: what stays stable, what gets appended, what goes to files, what errors remain visible, and how the agent keeps its goal in focus.

## 🗺️ One-Screen Map

| # | Principle | Core Problem | Manus-style Solution | Your Practical Meaning |
|---|---|---|---|---|
| 1 | ⚡ KV Cache First | Context prefix changes make agents slow and expensive | Stable prefix, append-only history, deterministic serialization | Don't put changing time/branch/memory inside system prompt |
| 2 | 🧰 Mask, Don't Remove | Dynamic tool lists break cache and confuse the model | Keep all tools fixed, only restrict what can be selected | `ALL_TOOLS` stays fixed; state controls allowed tools |
| 3 | 📁 File System = Memory | Logs/PDFs/DOM/diffs explode context | Store big data in files, keep only paths in context | `logs/`, `screenshots/`, `diffs/`, `corpus/` |
| 4 | 📝 Recitation | Long tasks drift away from the goal | Keep updating `todo.md` near context tail | Every major node updates/re-reads TODO |
| 5 | 🧯 Keep Errors | Clearing failures removes evidence | Keep stack traces and failed observations | Error logs help the agent self-correct |
| 6 | 🔁 Don't Few-Shot Yourself | Repeated patterns make model go autopilot | Add small structured variation | Avoid rigid repeated action templates |

## 1. ⚡ Design Around KV Cache

<div class="callout info">
<strong>What is happening?</strong><br>
Agent calls are usually long-input / short-output. The model repeatedly reads system prompt, tool definitions, and all prior actions/observations.
</div>

Agent loop:

```text
User Task
→ Model decides action
→ Tool runs
→ Observation returns
→ Append action + observation to context
→ Repeat
```

### ❌ Bad Pattern

```python
system_prompt = f"""
You are a bug-fixing agent.

Current time: {datetime.now()}
Current branch: {git_branch}
Current memory: {memory}
"""
```

Why bad?

- The prefix changes every step.
- Even a tiny change can invalidate cache after that point.
- More latency, more cost.

### ✅ Good Pattern

```python
SYSTEM_PROMPT = "You are a production bug-fixing agent. Follow the workflow strictly."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Current time: {now}. Current branch: {branch}."}
]
```

<div class="callout good">
<strong>Rule:</strong><br>
Keep the beginning stable. Put dynamic information later.
</div>

| Bad | Good |
|---|---|
| `system=f"time={now}"` | fixed `SYSTEM_PROMPT` |
| Rewriting old messages | append new message |
| Random JSON key order | sorted / deterministic JSON |
| Changing tool schema every node | fixed tool registry |

## 2. 🧰 Mask, Don't Remove

<div class="callout warn">
<strong>This is one of the most important production-agent lessons.</strong><br>
The tool list should usually stay stable. The agent's current state should control what tools are allowed, not physically remove tools from context every step.
</div>

### Example Tool Set

```python
ALL_TOOLS = [
    browser_open,
    browser_click,
    browser_screenshot,
    shell_run,
    file_read,
    file_write,
    git_diff,
    git_commit,
    slack_send,
]
```

### ❌ Bad: Remove Tools Dynamically

```python
if state == "browser":
    tools = [browser_open, browser_click, browser_screenshot]

if state == "shell":
    tools = [shell_run]
```

Why bad?

1. Tool definitions are near the front of context.
2. Changing them breaks KV cache.
3. History may contain tools that no longer exist in current context.
4. Model may hallucinate tool names or violate schemas.

Example confusion:

```text
History:
Action: browser_open

Current available tools:
shell_run
file_read

Model:
"Wait, browser_open existed before. Can I still use it?"
```

Possible bad output:

```json
{"tool": "browser_open_url", "args": {"url": "..."}}
```

`browser_open_url` does not exist.

### ✅ Good: Keep Tools Fixed, Mask Selection

```python
ALL_TOOLS = [
    browser_open,
    browser_click,
    browser_screenshot,
    shell_run,
    file_read,
    file_write,
    git_diff,
    git_commit,
    slack_send,
]

state_policy = {
    "browser": ["browser_"],
    "shell": ["shell_"],
    "editing": ["file_"],
    "git": ["git_"],
    "notify": ["slack_"],
}
```

<div class="callout key">
<strong>Mental Model</strong><br>
Remove = delete dishes from the menu every minute.<br>
Mask = menu stays stable, but the waiter says: "Right now you can only order from this section."
</div>

### Naming Matters

| Good Tool Names | Bad Tool Names |
|---|---|
| `browser_open` | `open` |
| `browser_click` | `clickThing` |
| `shell_run` | `execute` |
| `file_write` | `save` |
| `git_commit` | `commitCode` |

Prefixes make tool groups easy to constrain.

## 3. 📁 File System = Long-Term Memory

<div class="callout info">
<strong>Problem:</strong><br>
Real agents produce huge observations: webpage text, PDF text, DOM snapshots, console logs, screenshots, test outputs, git diffs.
</div>

### ❌ Bad: Put Everything Into Context

```text
500 lines console logs
3000 lines DOM snapshot
1000 lines test output
800 lines git diff
OCR text
PDF text
...
```

Result:

- Expensive
- Slow
- Context limit risk
- Model performance drops in long context
- Hard to know what is important later

### ✅ Good: Store Large Outputs as Files

```python
write_file("logs/console_error.log", console_output)
write_file("debug/dom_snapshot.html", dom_snapshot)
write_file("screenshots/login_failure.png", screenshot)
write_file("patches/current_diff.patch", git_diff)
```

Context only keeps references:

```text
Console log saved to logs/console_error.log
DOM snapshot saved to debug/dom_snapshot.html
Screenshot saved to screenshots/login_failure.png
```

When needed:

```python
read_file("logs/console_error.log")
```

<div class="callout good">
<strong>Rule:</strong><br>
Memory ≠ tokens. Memory = filesystem + paths + selective retrieval.
</div>

| Project | File Memory Design |
|---|---|
| Playwright bug-fix agent | `logs/`, `screenshots/`, `dom/`, `diffs/`, `test_results/` |
| AI Career Radar | `corpus/company_role_date/job.md`, `metadata.json`, `analysis.md` |
| Email agent | `threads/`, `inbound/`, `outbound/`, `errors/` |
| LangGraph experiments | `runs/`, `checkpoints/`, `observations/`, `todos/` |

## 4. 📝 Recitation = Attention Control

<div class="callout warn">
<strong>Problem:</strong><br>
Long agent tasks drift. The task goal may appear at the beginning, but after many tool calls it becomes buried in the middle.
</div>

### Bad

```text
Step 0:
Goal = fix login regression

Step 30:
Agent is reading package.json and may forget the original goal.
```

### Good: Maintain `todo.md`

```markdown
# TODO

[x] Reproduce the login bug
[x] Capture screenshot
[x] Collect console errors
[ ] Identify failing component
[ ] Patch the code
[ ] Run tests
[ ] Generate final summary
```

The agent periodically updates and re-reads it.

```python
update_file("todo.md", new_todo)
read_file("todo.md")
```

<div class="callout key">
<strong>Why it works:</strong><br>
The goal gets moved back near the end of context. Recent context is easier for the model to attend to.
</div>

### For LangGraph

```python
class AgentState(TypedDict):
    task: str
    todo_path: str
    current_step: str
    artifacts: list[str]
    errors: list[str]
```

## 5. 🧯 Keep the Wrong Stuff In

<div class="callout bad">
<strong>Bad instinct:</strong><br>
When something fails, clean the trace and retry silently.
</div>

### ❌ Bad

```python
try:
    run_tests()
except Exception:
    clear_history()
    retry()
```

Why bad?

- The model doesn't see why it failed.
- It may repeat the same mistake.
- The system loses useful debugging evidence.

### ✅ Good

Keep failed actions and observations:

```text
Action:
shell_run("npm test")

Observation:
ModuleNotFoundError: Cannot find module '@playwright/test'
```

Next step becomes obvious:

```python
shell_run("npm install -D @playwright/test")
```

<div class="callout good">
<strong>Rule:</strong><br>
Failures are not noise. Failures are learning signals.
</div>

| Failure Type | Keep It Where |
|---|---|
| Stack trace | `logs/stack_trace.log` |
| Test failure | `test_results/failure.txt` |
| Browser console error | `logs/browser_console.log` |
| Timeout | `logs/timeouts.log` |
| Bad tool call | context history + `errors/tool_error.json` |

## 6. 🔁 Don't Few-Shot Yourself

<div class="callout info">
<strong>Problem:</strong><br>
LLMs imitate patterns in context. In long workflows, the agent's own history becomes accidental few-shot examples.
</div>

### Bad Repeated Pattern

```text
Open file
Extract skills
Save summary

Open file
Extract skills
Save summary

Open file
Extract skills
Save summary
```

Eventually the model may go autopilot:

- Less careful reading
- Overgeneralization
- Hallucination
- Mechanical summaries

### Good: Add Small Structured Variation

```text
Opened: meta_job.md
Loaded: anthropic_role.md
Reading: openai_position.md
Parsing: google_job.md
```

Also vary:

- Observation format
- Ordering
- Summary template
- Wording

<div class="callout key">
<strong>Rule:</strong><br>
Too uniform context makes agents brittle.
</div>

### For AI Career Radar

Stable schema:

```json
{
  "company": "...",
  "role": "...",
  "skills": [],
  "agent_relevance": "...",
  "notes_path": "..."
}
```

Small variation can happen in natural-language observations, not in the data schema.

## 🧩 Direct Mapping to Your Projects

| Your Project | Manus Principle | Concrete Implementation |
|---|---|---|
| LangGraph bug-fix agent | Mask, don't remove | Global `ALL_TOOLS`; state decides allowed prefixes |
| Playwright automation | File system memory | Save screenshots, DOM, console logs, test outputs as files |
| AI Career Radar | External memory | `corpus/` folder with job post, metadata, extracted skills |
| Email agent | Keep errors | Preserve SendGrid/webhook/API failures in logs |
| Long-running agent flow | Recitation | Maintain `todo.md` throughout the graph |
| MCP tool ecosystem | Fixed tool schema | Avoid constantly changing exposed tool list mid-run |

## ✅ Implementation Checklist

<div class="callout good">
Use this checklist when designing any serious agent.
</div>

| Check | Question |
|---|---|
| ⬜ Stable prefix | Does system prompt stay fixed across steps? |
| ⬜ Append-only context | Am I avoiding rewriting old history? |
| ⬜ Deterministic serialization | Are JSON keys/order stable? |
| ⬜ Fixed tool registry | Am I avoiding dynamic tool removal? |
| ⬜ Tool masking | Does state restrict allowed tools without changing schema? |
| ⬜ File memory | Are big outputs stored as files? |
| ⬜ Restorable compression | If I remove content from context, can agent recover it via path/URL? |
| ⬜ TODO recitation | Does agent periodically refresh the goal? |
| ⬜ Error retention | Are failures preserved for self-correction? |
| ⬜ Anti-autopilot variation | Am I avoiding overly repetitive traces? |

## 🧠 Final Memory Hooks

| Hook | Meaning |
|---|---|
| ⚡ Stable Prefix = Cheap Agent | Cache works only if the beginning stays stable |
| 🧰 Fixed Tools = Stable Agent | Don't mutate tool schema every step |
| 📁 Filesystem > Long Context | Store large observations outside the model window |
| 📝 TODO = Attention Control | Keep the goal near the end of context |
| 🧯 Failures = Learning Signals | Errors help the model avoid repeating mistakes |
| 🔁 Uniform Context = Fragile Agent | Too much repetition makes the agent mechanical |